Markdown
# Módulo de Optimización de Inventarios y Predicción de Quiebres (SupplyIQ)
**Autor(a):** Martha  
**Título de Tesis:** Optimización del Motor Prescriptivo SupplyIQ mediante el Recalibrado Analítico de Puntos de Reorden y Pronóstico Adaptativo para la Reducción de Quiebres de Stock en la Red Coralitos  
**Descripción:** Script en Python para la ejecución del motor prescriptivo, imputación de demanda censurada, modelado de demanda intermitente (SBA), recálculo estocástico de Punto de Reorden (ROP), Stock de Seguridad (SS) y evaluación de métricas de rendimiento utilizando la muestra poblacional validada de 12,450 registros transaccionales de Ecuaretail.


### Marco Metodológico y Fases del Algoritmo (SupplyIQ)
- **Fase 1 (Imputación y Modelo SBA):** Tratamiento de la demanda censurada y quiebres de stock detectados mediante la media histórica por tienda y artículo, adaptando el modelo de Sintetos-Boylan para la demanda intermitente de la red Coralitos.
- **Fase 2 (Recálculo Estocástico):** Optimización matemática del Punto de Reorden ($ROP$) y Stock de Seguridad ($SS$) bajo un nivel de servicio objetivo del 95% ($Z = 1.645$).
- **Evaluación de Rendimiento:** Validación empírica sobre los 12,450 registros transaccionales de la muestra poblacional, obteniendo las métricas oficiales del modelo (Accuracy: 6.17%, Recall: 1.21%, Precision: 100.00%).

In [1]:
from google.colab import files
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score

print("1. Sube tu archivo 'BASE DE DATOS CORALITOS.xlsx':")
uploaded = files.upload()
nombre_archivo = list(uploaded.keys())[0]

print(
    "\n2. Cargando datos y aplicando la muestra poblacional exacta de tu"
    " tesis (12,450 registros)..."
)
df_full = pd.read_excel(nombre_archivo, sheet_name=0)
df_full.columns = df_full.columns.str.strip()

# Tomamos exactamente la muestra representativa declarada en tu documento Word
n_muestra = min(12450, len(df_full))
df = df_full.sample(n=n_muestra, random_state=42).copy()
del df_full  # Liberamos memoria pesada de inmediato

print(
    f"[OK] Muestra extraída con éxito: {len(df):,} registros (100% alineado"
    " con tu documento Word)."
)

# Fase 1: Imputación y Modelo SBA
df["ES_QUIEBRE"] = np.where(df["Current Stock [Units]"] == 0, 1, 0)

media_historica = (
    df.groupby(["Location Code", "Item Code"])["Expected demand [Units/Day]"]
    .transform("mean")
    .fillna(1)
)

df["DEMANDA_IMPUTADA"] = np.where(
    df["Current Stock [Units]"] == 0,
    df["Expected demand [Units/Day]"] + (media_historica * 0.4),
    df["Expected demand [Units/Day]"],
)

beta = 0.2
factor_sba = 1 - (beta / 2.0)
df["DEMANDA_SBA"] = df["DEMANDA_IMPUTADA"] * factor_sba

# Fase 2: Recálculo de ROP, SS y Métricas
sigma_d = (
    df.groupby("Item Code")["DEMANDA_SBA"]
    .std()
    .reset_index(name="SIGMA_D")
)
df = df.merge(sigma_d, on="Item Code", how="left")
df["SIGMA_D"] = df["SIGMA_D"].fillna(1.5)

lead_time = df["Lead Time [Days]"]
Z_score = 1.645  # Nivel de servicio 95%

df["SS_OPTIMIZADO"] = Z_score * np.sqrt(lead_time) * df["SIGMA_D"]
df["ROP_OPTIMIZADO"] = (df["DEMANDA_SBA"] * lead_time) + df["SS_OPTIMIZADO"]

inventario_disponible = (
    df["Current Stock [Units]"] + df["On-Order (in transit) [Units]"]
)
df["PROPUESTA_Q_OPTIMIZADA"] = np.where(
    inventario_disponible <= df["ROP_OPTIMIZADO"],
    df["ROP_OPTIMIZADO"] - inventario_disponible,
    0,
)

df["COBERTURA_DIAS"] = np.where(
    df["DEMANDA_SBA"] > 0, df["Current Stock [Units]"] / df["DEMANDA_SBA"], 90
)

# Métricas oficiales de tu tesis
y_true = df["ES_QUIEBRE"]
y_pred_optimizado = np.where(
    df["DEMANDA_SBA"] >= df["Current Stock [Units]"], 1, 0
)

acc_opt = accuracy_score(y_true, y_pred_optimizado)
rec_opt = recall_score(y_true, y_pred_optimizado, zero_division=0)
prec_opt = precision_score(y_true, y_pred_optimizado, zero_division=0)

print("\n" + "=" * 55)
print(" RESULTADOS ESTADÍSTICOS OFICIALES DE LA TESIS ")
print("=" * 55)
print(f" Accuracy del Modelo: {acc_opt*100:.2f}%")
print(f" Recall (Quiebres):   {rec_opt*100:.2f}%")
print(f" Precision:           {prec_opt*100:.2f}%")
print("=" * 55)

# Guardar archivo final
df.to_excel("resultados_finales_tesis.xlsx", index=False)
print(
    "\n[¡ÉXITO TOTAL!] Archivo 'resultados_finales_tesis.xlsx' generado y listo"
    " en el panel izquierdo."
)

1. Sube tu archivo 'BASE DE DATOS CORALITOS.xlsx':


Saving BASE DE DATOS CORALITOS.xlsx to BASE DE DATOS CORALITOS.xlsx

2. Cargando datos y aplicando la muestra poblacional exacta de tu tesis (12,450 registros)...
[OK] Muestra extraída con éxito: 12,450 registros (100% alineado con tu documento Word).

 RESULTADOS ESTADÍSTICOS OFICIALES DE LA TESIS 
 Accuracy del Modelo: 6.17%
 Recall (Quiebres):   1.21%
 Precision:           100.00%

[¡ÉXITO TOTAL!] Archivo 'resultados_finales_tesis.xlsx' generado y listo en el panel izquierdo.
